In [ ]:
%load_ext autoreload
%autoreload 2
import os
os.environ["OPENBLAS_NUM_THREADS"] = "4"
import sys, re, torch, time, numpy as np, pandas as pd, scanpy as sc, de_test
from de_test import fitGAM, associationTest, predictSmooth, startVsEndTest
import matplotlib.pyplot as plt
import seaborn as sns

os.chdir("/ssd/users/Wergillius/Project/PINN_dynamics")

# Load data
adata = sc.read_h5ad("data/tom_pos.h5ad")
adata_raw = adata.raw.to_adata()

In [ ]:
# Subset to Megakaryocyte lineage using pseudotime
mkpar = pd.read_csv('./PD_model/clu_7/tables/table_all_parameters_clu_7.csv')
mk = adata[mkpar['V1'], :]
mk = mk[mk.obs['data_type'] != 'SS2',:]

In [ ]:
# Prepare input data
counts = mk[:, mk.var_names].X.toarray()
pseudotime = mkpar['dpt_pseudotime'].values

# Run GAM fitting
gam_fit = fitGAM(count_matrix=counts, cell_time=pseudotime, n_knots=7)

In [ ]:
# Association test
de_results = associationTest(gam_fit)
print(de_results.head())

In [ ]:
# Smooth expression prediction
smooth_expr = predictSmooth(gam_fit, nPoints=100)
print(smooth_expr.head())

In [ ]:
# Start vs end test
svs_results = startVsEndTest(gam_fit)
print(svs_results.head())

In [ ]:
# Plot top correlated genes with drift
drift = mkpar['drift']
x = np.linspace(0, 1, 100)
af = interp1d(pseudotime, drift, fill_value='extrapolate')
y = af(x)
ddrift = np.gradient(y, x)

# Get top TFs
allTFs = pd.read_csv('./data/genesets/TFs_Ravasi2010short.csv')['Symbol (Mouse)'].unique()
smooth_tfs = smooth_expr[smooth_expr.columns.intersection(allTFs)]

# Compute correlations
correlations = {}
for gene in smooth_tfs.columns:
    corr, _ = pearsonr(smooth_tfs[gene], ddrift)
    correlations[gene] = corr

top_tfs = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)[:10]
print('Top correlated TFs:', top_tfs)